# History from nsepy

In [1]:
from nsepy import get_history
from datetime import date, datetime, timedelta
import pandas as pd

#inputs
symbol = 'SBIN'
is_index=False
days = 365
end = datetime.now().date()
start = end-timedelta(days=days)

# call
data = get_history(symbol=symbol, start=start, end=end, index=is_index).reset_index()
data.Date = data.Date.apply(pd.to_datetime)

# check
data.Date.iloc[-1] - data.Date.iloc[0]


Timedelta('361 days 00:00:00')

In [ ]:
data

In [ ]:
# Trying my own history!!

import datetime

import requests
from bs4 import BeautifulSoup
import pandas as pd

symbol = 'nifty'

session = requests.Session()

period=364 # max for 364 days!
end = datetime.datetime.now()
start = end - datetime.timedelta(days=period)


In [ ]:
# For NIFTY index

from pathlib import Path
import yaml
import logging

url='http://www1.nseindia.com/products/dynaContent/equities/indices/historicalindices.jsp'

# Header from yml files
root = Path().cwd() 
with open(root / "conf" / "config.yml", 'r') as f:
    config = yaml.safe_load(f)

headers = config['headers']
try:
    indexType = config['index_for_history'][symbol.upper()]
except KeyError as e:
    logging.error(f"Please choose amongst symbols {list(config['index_for_history'].keys())}")
    # sys.exit(0)
    


In [ ]:
params={'indexType': indexType, 'fromDate': start.strftime('%d-%m-%Y'), 'toDate': end.strftime('%d-%m-%Y')}

resp = session.get(url=url, params=params, headers=headers)
soup = BeautifulSoup(resp.text, 'lxml')
table = soup.find_all('table')

try:
    df = pd.read_html(str(table))[0]
except ValueError as e:
    print(f"illformed table!! {e}")

# cleaning up the table
cols = [list(c)[-1] for c in df.columns]
df.columns = cols

df[df.columns[0]] = pd.to_datetime(df.Date, errors='coerce') # coerces last row to NaT
df = df.dropna()
df[df.columns[1:]] = df[df.columns[1:]].apply(pd.to_numeric) # converts non-date columns to numeric


In [ ]:
df

In [ ]:
df.tail(1).Date.iloc[0] - df.head(1).Date.iloc[0]

In [ ]:
end-start

In [ ]:
end